Notebook goal:

Generate unique cluster IDs with cluster IDs of more similar cell types being closer together

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path 

import numpy as np
import pandas as pd
import scanpy as sc
from dotenv import dotenv_values

In [3]:
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])
CLUSTERING_DIR = DATADIR / "5_clustering"
MODELDIR = CLUSTERING_DIR / 'all_control'
DATASETS_DIR = MODELDIR / "datasets"

In [4]:
ADATA_FP = DATASETS_DIR / 'raw_embed_propagated.h5ad'

IMMUNE_FP = CLUSTERING_DIR / "immune_labelmat.csv"
VASC_FP = CLUSTERING_DIR / "vasc_labelmat.csv"
ASTRO_FP = CLUSTERING_DIR / "astro_epen_labelmat.csv"
OLIGO_FP = CLUSTERING_DIR / "oligo_labelmat.csv"
NEURO_FP = CLUSTERING_DIR / "neuron_labelmat.csv"

LABEL_OUT_FP = CLUSTERING_DIR / "final_labelmat.csv"

### Let's load the anndata

In [5]:
adata = sc.read_h5ad(ADATA_FP)

In [6]:
#630146 + 292309 == len(adata)

### Let's load all labelmats, harmonize them and make labels unqiue for level and cell type (preliminary)

In [7]:
def harmonize_cols(df, suffix=""):
    for c in df.columns:
        i = c[-1]

        col = df[c]
        if isinstance(col.iloc[0], float):
            col = col.astype(int)

        df[c] = col.astype(str) + suffix

    return df

In [8]:
neuro = pd.read_csv(NEURO_FP, index_col=0)
neuro = neuro.loc[~neuro.isna().any(axis=1), :]
neuro=harmonize_cols(neuro, suffix="-Neuro")
#neuro

In [9]:
immune = pd.read_csv(IMMUNE_FP, index_col=0) 
immune = immune.loc[~immune.isna().any(axis=1), :]
immune = harmonize_cols(immune, suffix="-Immune")
immune["L1"] = "NN-Immune"
#immune

In [10]:
vasc = pd.read_csv(VASC_FP, index_col=0) 
vasc = vasc.loc[~vasc.isna().any(axis=1), :]
vasc = harmonize_cols(vasc, suffix="-Vascular")
vasc["L1"] = "NN-Vascular"
#vasc

In [11]:
oligo = pd.read_csv(OLIGO_FP, index_col=0) 
oligo = oligo.loc[~oligo.isna().any(axis=1), :]
oligo = harmonize_cols(oligo, suffix="-Oligo")
oligo["L1"] = "NN-Oligo"
#oligo

In [12]:
astro = pd.read_csv(ASTRO_FP, index_col=0) 
astro = astro.loc[~astro.isna().any(axis=1), :]
astro = harmonize_cols(astro, suffix="-Astro-Epen")
astro["L1"] = "NN-Astro-Epen"
#astro

In [13]:
labelmat_df = pd.concat([neuro, immune, vasc, oligo, astro], axis=0)

In [14]:
# labelmat_df.loc[labelmat_df.L5.isna(), "L5"] = labelmat_df.loc[labelmat_df.L5.isna(), "L4"].apply(lambda x: "NN5" + x[3:])
labelmat_df.loc[labelmat_df.L5.isna(), "L5"] = labelmat_df.loc[labelmat_df.L5.isna(), "L4"]

In [15]:
labelmat_df

,L1,L2,L3,L4,L5
SC1-01_SAMN18754353_AAACCCACAACCAGAG,Glut-Neuro,405-Neuro,3662569-Neuro,1823023149-Neuro,1823023149-Neuro
SC1-01_SAMN18754353_AAACCCAGTCATCTAG,Glut-Neuro,61775-Neuro,3484-Neuro,251692-Neuro,241087849-Neuro
SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,Glut-Neuro,405-Neuro,3465026-Neuro,397831525-Neuro,397831525-Neuro
SC1-01_SAMN18754353_AAACGAAGTGCGGCTT,Glut-Neuro,405-Neuro,3662569-Neuro,2602618-Neuro,2602618-Neuro
SC1-01_SAMN18754353_AAACGAATCGCATAGT,Glut-Neuro,405-Neuro,3275518-Neuro,141818058-Neuro,20519226898-Neuro
...,...,...,...,...,...
SC2-25_SAMN46505609_TTTGTCTAGCAAGACA,NN-Astro-Epen,54-Astro-Epen,11-Astro-Epen,11-Astro-Epen,11-Astro-Epen
SC2-25_SAMN46505609_TTTGTGAAGCTTACTT,NN-Astro-Epen,54-Astro-Epen,11-Astro-Epen,11-Astro-Epen,11-Astro-Epen
SC2-25_SAMN46505609_TTTGTGGCAATAGTCT,NN-Astro-Epen,54-Astro-Epen,11-Astro-Epen,11-Astro-Epen,11-Astro-Epen
SC2-25_SAMN46505609_TTTGTGGCACAAAGCG,NN-Astro-Epen,20-Astro-Epen,52-Astro-Epen,52-Astro-Epen,52-Astro-Epen


In [16]:
adata = adata[labelmat_df.index, :].copy()
adata.obs[[f"L{i}" for i in range(1,6)]] = labelmat_df

In [17]:
adata

AnnData object with n_obs × n_vars = 920995 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'cluster_id_label', 'supertype_id_label', 'subclass_id_label', 'class_id_label', 'C66_named', 'C185_named', 'C286_named', 'C465_named', 'ABC_class_propagated', 'ABC_class_confidence', 'to_be_removed', 'IsNeuronal', 'major_features', 'major_features_propagated', 'major_features_confidence', 'ABC_supertype_propagated', 'ABC_supertype_confidence', 'ABC_subclass_propagated', 'ABC_subclass_confidence', 'ABC_cluster_propagated', 'ABC_cluster_confidence', 'L1', 'L2', 'L3', 'L4', 'L5'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    uns: 'neighbors'

### Let's create a 1D umap to quantify similarity 

In [18]:

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    use_rep="X_embed"
)
    
sc.tl.umap(adata, key_added='X_umap_1d', n_components=1)


## Let's get the IDs

In [19]:

# Separating neurons and non-neurons artificially
# find max value of 1d UMAP and add non-neuronal scores to separate them fully / artificially
# This makes non-neurons last in ID order
non_neuronal_mask = adata.obs['L1'].str.startswith('NN').values
max_value = np.max(adata.obsm['X_umap_1d'])
adata.obsm['X_umap_1d'][non_neuronal_mask] = max_value + adata.obsm['X_umap_1d'][non_neuronal_mask]

# Extract labelmat as numpy arr
keys = ['L1', 'L2', 'L3', 'L4', 'L5']
labelmat = np.asarray(adata.obs[keys])

# Get all unique labels (ignorant of level)
unique_labels, inverse = np.unique(labelmat, return_inverse=True)

# Compute 1D umap centroid for each label
centroids = np.zeros_like(unique_labels, dtype=np.float32)
for i, label in enumerate(unique_labels):
    mask = np.any(labelmat == label, axis=1)
    centroids[i] = np.mean(adata.obsm['X_umap_1d'][mask])

# Map these back to get a new matrix with centroids instead of labels
centroid_labelmat = centroids[inverse].reshape(labelmat.shape)

# Get all unique paths (rows)
centroid_pathmat, path_indices = np.unique(centroid_labelmat, return_index=True, axis=0)
pathmat = labelmat[path_indices] # mirror transform of line above

# Sort each path from L1 ---> L5 (note: lexsort works backwards - counterintuitive I know)
path_order = np.lexsort(centroid_pathmat.T[::-1])
ordered_pathmat = pathmat[path_order]

# Order unique labels by first appearance in 1D umap order so label --> index
occurence = []
for v in ordered_pathmat.T.ravel():
    if v not in occurence:
        occurence.append(v)

# IDs are then their order of occurence and mapped back to original labelmat
idmat = np.array([[occurence.index(x) for x in row] for row in labelmat])
idmat += 1 # add 1 as to start idx at 1 and not 0

idmat

array([[   1,   12,  112,  403,  403],
       [   1,   13,  115,  418, 1043],
       [   1,   12,  109,  392,  392],
       ...,
       [   3,   83,  349,  349,  349],
       [   3,   82,  344,  344,  344],
       [   3,   83,  349,  349,  349]], shape=(920995, 5))

In [20]:

# Formatted IDs for greater consistency so 1 --> 01 where max id in that level is 2 digits etc
idmat_formatted = idmat.astype(str)
for i in range(idmat_formatted.shape[1]):
    max_id = np.max(idmat[:, i])
    n_digits = np.floor(np.log10(max_id)).astype(int) + 1
    idmat_formatted[:, i] = [f"{x:0{n_digits}d}" for x in idmat[:, i]]

idmat_formatted


array([['1', '12', '112', '0403', '0403'],
       ['1', '13', '115', '0418', '1043'],
       ['1', '12', '109', '0392', '0392'],
       ...,
       ['3', '83', '349', '0349', '0349'],
       ['3', '82', '344', '0344', '0344'],
       ['3', '83', '349', '0349', '0349']],
      shape=(920995, 5), dtype='<U21')

### Now put everything in df, insert level- and cell type-dependent info

In [40]:
label_df = pd.DataFrame(data=idmat_formatted, index=adata.obs_names, columns=[f"L{i}" for i in range(1,6)])

In [41]:
gb_mask = labelmat_df.L1.str.contains("GABA")
gl_mask = labelmat_df.L1.str.contains("Glut")

for i in range(1,6):
    label_df.loc[non_neuronal_mask, f"L{i}"] = label_df.loc[non_neuronal_mask, f"L{i}"].apply(lambda x: "NN" + str(i) + "-" + x)
    label_df.loc[gb_mask, f"L{i}"] = label_df.loc[gb_mask, f"L{i}"].apply(lambda x: "Gb" + str(i) + "-" + x)
    label_df.loc[gl_mask, f"L{i}"] = label_df.loc[gl_mask, f"L{i}"].apply(lambda x: "Gl" + str(i) + "-" + x)

In [43]:
label_df.loc[non_neuronal_mask, "L1"] = (
    label_df.loc[non_neuronal_mask, "L1"].astype(str)
    + " "
    + labelmat_df.loc[non_neuronal_mask, "L1"].str.replace(r"^NN-", "", regex=True)
)

In [44]:
label_df.loc[~non_neuronal_mask, "L1"] = (
    label_df.loc[~non_neuronal_mask, "L1"].astype(str)
    + " "
    + labelmat_df.loc[~non_neuronal_mask, "L1"].str.replace(r"-Neuro", "", regex=True)
)

In [45]:
label_df

,L1,L2,L3,L4,L5
SC1-01_SAMN18754353_AAACCCACAACCAGAG,Gl1-1 Glut,Gl2-12,Gl3-112,Gl4-0403,Gl5-0403
SC1-01_SAMN18754353_AAACCCAGTCATCTAG,Gl1-1 Glut,Gl2-13,Gl3-115,Gl4-0418,Gl5-1043
SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,Gl1-1 Glut,Gl2-12,Gl3-109,Gl4-0392,Gl5-0392
SC1-01_SAMN18754353_AAACGAAGTGCGGCTT,Gl1-1 Glut,Gl2-12,Gl3-112,Gl4-0405,Gl5-0405
SC1-01_SAMN18754353_AAACGAATCGCATAGT,Gl1-1 Glut,Gl2-12,Gl3-111,Gl4-0398,Gl5-1037
...,...,...,...,...,...
SC2-25_SAMN46505609_TTTGTCTAGCAAGACA,NN1-3 Astro-Epen,NN2-83,NN3-349,NN4-0349,NN5-0349
SC2-25_SAMN46505609_TTTGTGAAGCTTACTT,NN1-3 Astro-Epen,NN2-83,NN3-349,NN4-0349,NN5-0349
SC2-25_SAMN46505609_TTTGTGGCAATAGTCT,NN1-3 Astro-Epen,NN2-83,NN3-349,NN4-0349,NN5-0349
SC2-25_SAMN46505609_TTTGTGGCACAAAGCG,NN1-3 Astro-Epen,NN2-82,NN3-344,NN4-0344,NN5-0344


In [46]:
# checking that hierarchy is maintained, i.e., each L5 has exactly one path moving from L1 to L4

check = (
    label_df
    .groupby('L5')[['L1', 'L2', 'L3', 'L4']]
    .nunique()
)

invalid = check[(check > 1).any(axis=1)]

assert len(invalid) == 0, "Hierarchy violation found"

In [48]:
for c in label_df.columns:
    print(label_df[c].nunique())

6
88
286
733
962


### Let's save

In [49]:
label_df.to_csv(LABEL_OUT_FP)